In [1]:
%pip install -q pyspark

In [2]:
from google.colab import files
from pyspark.sql import SparkSession

uploaded = files.upload()
filename = next(iter(uploaded))

spark = SparkSession.builder.master("local[2]").appName("ClaimsPipeline").getOrCreate()
claims = spark.read.csv(filename, header=True)
print("Rows read:", claims.count())

Saving claims_2025-12-22.csv to claims_2025-12-22.csv
Rows read: 141


In [3]:
from google.colab import files
from pathlib import Path
from zipfile import ZipFile

uploaded = files.upload()

with ZipFile("landing.zip") as archive:
    archive.extractall("/content/claims_input")

csv_files = list(Path("/content/claims_input").rglob("*.csv"))
print(f"Found {len(csv_files)} CSVs")

Saving landing.zip to landing.zip
Found 107 CSVs


In [6]:
from pathlib import Path
from pyspark.sql import functions as F

landing = Path("/content/claims_input")
bronze = Path("/content/bronze")
bronze.mkdir(exist_ok=True)

csv_files = sorted(landing.rglob("*.csv"))
assert len(csv_files) == 107, f"Expected 107 CSVs, found {len(csv_files)}"

written = 0
skipped = 0

for source in csv_files:
    destination = bronze / source.stem
    staging = bronze / f".{source.stem}.tmp"

    if (destination / "_SUCCESS").is_file():
        skipped += 1
        continue

    batch_id = source.stem.split("_")[1]
    rows = (
        spark.read.csv(str(source), header=True, inferSchema=False)
        .withColumn("source_file", F.lit(source.name))
        .withColumn("ingest_time", F.current_timestamp())
        .withColumn("batch_id", F.lit(batch_id))
    )

    rows.write.mode("overwrite").parquet(str(staging))
    staging.rename(destination)
    written += 1

print(f"Files written: {written} | Files skipped: {skipped}")

Files written: 0 | Files skipped: 107


In [7]:
def read_bronze(pattern):
    paths = [str(path) for path in sorted(bronze.glob(pattern))]
    return spark.read.option("mergeSchema", "true").parquet(*paths)

bronze_claims = read_bronze("claims_*")
bronze_transactions = read_bronze("transactions_*")

print("Bronze claim rows:", bronze_claims.count())
print("Bronze transaction rows:", bronze_transactions.count())

Bronze claim rows: 9769
Bronze transaction rows: 89474


In [ ]:
from pathlib import Path
from google.colab import files
from pyspark.sql import functions as F

if not Path("silver.py").is_file():
    print("Choose src/silver.py from the project folder.")
    files.upload()

from silver import clean_claims, clean_transactions, write_silver

silver_path = "/content/silver"
checked_claims = clean_claims(bronze_claims)
checked_transactions = clean_transactions(bronze_transactions)

print(f"Claim rows removed as duplicates: {bronze_claims.count() - checked_claims.count():,}")
for table, checked in [("claims", checked_claims), ("transactions", checked_transactions)]:
    valid, invalid = write_silver(table, checked, silver_path)
    print(f"{table}: {valid:,} valid | {invalid:,} quarantined")


In [10]:
silver_claims = spark.read.parquet("/content/silver/claims")
silver_transactions = spark.read.parquet("/content/silver/transactions")

claim_ids = silver_claims.select(F.col("Id").alias("CLAIMID"))
unmatched = silver_transactions.join(claim_ids, on="CLAIMID", how="left_anti")

print("Transactions without a valid claim:", unmatched.count())

Transactions without a valid claim: 14


In [11]:
from google.colab import files

claims = spark.read.parquet("/content/silver/claims").selectExpr(
    "Id AS CLAIM_ID",
    "PATIENTID AS PATIENT_KEY",
    "PROVIDERID AS PROVIDER_ID",
    "substr(SERVICEDATE, 1, 10) AS SERVICE_DATE",
    "STATUS1 AS CLAIM_STATUS",
    "batch_id AS BATCH_ID",
)

transactions = spark.read.parquet("/content/silver/transactions").selectExpr(
    "ID AS TRANSACTION_ID",
    "CLAIMID AS CLAIM_ID",
    "TYPE AS TRANSACTION_TYPE",
    "amount_decimal AS AMOUNT",
    "try_cast(PAYMENTS AS DECIMAL(18, 2)) AS PAYMENT_AMOUNT",
    "batch_id AS BATCH_ID",
)

for name, table in [("silver_claims", claims), ("silver_transactions", transactions)]:
    path = f"/content/{name}.csv"
    table.toPandas().to_csv(path, index=False)
    files.download(path)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [12]:
%%writefile /content/test_silver.py
import pytest
from pyspark.sql import SparkSession


@pytest.fixture(scope="module")
def spark():
    return SparkSession.builder.getOrCreate()


def test_silver_claims(spark):
    claims = spark.read.parquet("/content/silver/claims")
    assert claims.count() == 9_227
    assert claims.select("Id").distinct().count() == 9_227
    assert claims.filter("Id IS NULL").count() == 0


def test_silver_transactions(spark):
    transactions = spark.read.parquet("/content/silver/transactions")
    assert transactions.count() == 89_473
    assert transactions.filter(
        "TYPE IN ('CHARGE', 'TRANSFERIN') AND amount_decimal IS NULL"
    ).count() == 0


def test_quarantine(spark):
    assert spark.read.parquet("/content/silver/quarantine/claims").count() == 1
    assert spark.read.parquet("/content/silver/quarantine/transactions").count() == 1

Writing /content/test_silver.py


In [13]:
import pytest

pytest.main(["-q", "/content/test_silver.py"])

...                                                                      [100%]
3 passed in 3.02s


<ExitCode.OK: 0>